# പാഠം 18 (തുടർന്ന്): ഒരു *മനുഷ്യൻ* പ്രവർത്തനം അനുമോദിച്ചത证明ിക്കുന്ന സ്വീകരണങ്ങൾ  

ഈ പാഠം **എജന്റ്** എന്ത് ചെയ്തു എന്നതും **ഗേറ്റു** എന്ത് തീരുമാനിച്ചു എന്നതും തെളിയിക്കുന്നു. ഈ നോട്ട് ബുക്ക് կորുന്ന പകുതി ավելացնում է՝ ապացույց, որ **նշված մարդը** հաստատեց **ճշգրիտ** գործողությունը — ամբողջ կանոնավոր գործողության առանձին, մարդու մոտ առկա ստորագրություն, որը ստուգվել է օֆլայն։  

Երկու արհեստանյութերն էլ այստեղ օգտագործում են նույն պայուսակի ձևը, ինչ դասագրքի վերագրականները՝ հարթ մեսիջով `type` դաշտով, ստորագրված է Ed25519-ով ուղղակի կանոնավոր JCS բայթերի վրա՝ հարակցված մի կառուցողական `signature` օբյեկտով (և դուրս թողնված է ստորագրված բայթերից): Միաձայնման ընդունումը նոր `type` է (`human.approval.v1`), գործողության տեսակի կողքին, այնպես որ `verify_chain`-ը մի ծածկում է երկու արհեստանյութերի տեսակները նույն կոդային ուղուց, որը դուք կազմել եք հիմնական նոթբուքում։ Այս մարդկային հաստատման ընդունումը ուսումնամեթոդական կազմություն է, որը սահմանված է այստեղ, այլ ոչ թե `draft-farley-acta-signed-receipts`-ով սահմանված ընդունման տեսակ։  

Հիմնական նոթբուքի դեմոն ստուգչից մի համազարկ բարելավում՝ ստուգիչը այստեղ խորհրդանշում է `signature.key_id`-ը **կպցված բանալի գրանցամատյանին** ընդդեմ հասարակ բանալիի վարկածի, որը ներառված է ընդունման մեջ։ Սա է արտադրական դիրքը, որը դասագրքի սեփական ստուգաթերթը խորհուրդ է տալիս ("հրապարակել ստուգման հանրային բանալին"), և սա է, որ կեղծագրությունը դարձնում է մերժում՝ ոչ թե սեփական բանալին վերցնելու շրջանցում։  

Այս նոթբուքը սովորեցնում է օրենքը՝ **ստորագրված հաստատումն ինքնին իշխանություն չէ։** Իշխանությունը գոյություն ունի միայն եթե հաստատման ընդունումը և գործողության ընդունումը դեռ կապված են նույն կանոնավոր գործողության հետ, կատարման պահին, գործող քաղաքականության տարբերակով, բանալով և ժամկետով, որոնք դեռ գործող են, և հաստատում, որը դեռ սպառված չէ։ Յուրաքանչյուր ձախողում մերժվում է հատուկ պատճառով, այնպես որ կարող եք տարբերակել *իշխանությունը հնացած է* և *կատարված գործողությունը փոխվել է*։  


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## കൃത്യമായ പ്രവർത്തനം

അംഗീകാരത്തിന്റെ ഘടകം **സാന്ദർഭിക പ്രവർത്തന വസ്തു** ആണ് — "റിഫണ്ട് അംഗീകരിക്കുക" പോലുള്ള അസ്പഷ്ടമായ ലേബലല്ല, പക്ഷേ കൃത്യമായ, പൂർണ്ണവിവരമുള്ള പ്രവർത്തനമാണ്. മുഴുവൻ വസ്തുവും (അതിൽ നിന്ന് ഒരു ഡയറസ്റ്റ് സമ്പാദിക്കുകയും) ഒപ്പിടുന്നത് പിന്നീട് മനുഷ്യൻ ഇതേ അംഗീകരിച്ചു, മറ്റു ഒന്നും അല്ലെന്ന് തെളിയിക്കാനും സഹായിക്കുന്നു.


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## ഒരു എന്വലോപ്പ്, രണ്ട് അതോറിട്ടികൾ

ഓരോ റെസിപ്പ്റ്റുമാണ് പാഠത്തിന്റെ എന്വലോപ്പ്: ഒരു ഫ്ലാറ്റ് പേയ്ലോഡ് `type` ഫീൽഡോടുകൂടി, കൂടാതെ `signature` ഒബ്‌ജക്റ്റ് (`alg`, `sig`, `key_id`) ആണ് അടങ്ങിയത്, ഇത് ഒപ്പ് വെച്ച ബൈറ്റുകളുടെ ഭാഗമല്ല. `verify_envelope` ഇരുഭേദങ്ങളുടെയും പൊതുവായ ഘടനാത്മക + ഒപ്പ് പരിശോധനയാണ്; അത് പരിഹരിക്കുന്നത് `signature.key_id` ഉപയോഗിച്ച് ഏത് **പിന്നഡുള്ള കീ രജിസ്ട്രിയാണെന്ന്** വേരിയാൻസുകൾ വേർതിരിക്കുന്നത്:

- **അപ്രൂവൽ റെസിപ്പ്റ്റ്** (`human.approval.v1`) — നാമം പറഞ്ഞ അപ്രൂവർ, പൂർണ്ണ കാനോണിക്കൽ ആക്ഷൻ **മറ്റും അതിന്റെ ഡൈജസ്റ്റ്**, `policy_version`, പുറപ്പെടുവിക്കൽ + കാലഹരണ തീയതികൾ. ഒരിക്കൽ മാത്രം ഉപയോഗം ചെയിൻ ലെവലിൽ ട്രാക്ക് ചെയ്യപ്പെടുന്നു.
- **ആക്ഷൻ റെസിപ്പ്റ്റ്** (`agent.action.v1`) — ഏജന്റ് ഐഡന്റിറ്റി, `run_id`, അതേ കാനോണിക്കൽ ആക്ഷൻ **ഡൈജസ്റ്റ്**, നിർവഹണ ഫലം + ടൈംസ്റ്റാമ്പ്, കൂടാതെ `parent_approval_ref`: അപ്രൂവലിന്റെ `receipt_hash`, പാഠത്തിന്റെ ചെയിന്റെ `previous_receipt_hash` എന്ന സമാനമായ конവെൻഷൻ.

പൊതുവായ `action_digest` ഫീൽഡ് ആണ് ബന്ധം ആശ്രയിക്കുന്നത്. `key_id` ഒപ്പ് ഒബ്‌ജക്റ്റിൽ ഒരു ലുക്ക് അപ്പ് സൂചനയായാണ് ഉള്ളത്: ഇതിനെ വിവിധ പിന്നഡുള്ള കീയിൽ തിരിയിക്കുന്നത് ഒപ്പ് പരിശോധന പരാജയപ്പെടാൻ കാരണമാകുന്നു, അതുകൊണ്ട് ഇത് ഒന്നും നൽകുന്നില്ല.


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the canonical JCS
    bytes directly; the signature object is NOT part of the signed bytes."""
    canonical = canonicalize(payload)
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(canonical).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized and passed directly to Ed25519.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        canonical = canonicalize(payload)
        VerifyKey(b64url_decode(pub)).verify(canonical, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: ബന്ധം യഥാർത്ഥത്തിൽ തീരുമാനിക്കുന്നത്

`verify_chain` രണ്ട് ഒപ്പിടൽ പരിശോധനകളുടെ സൌകര്യമായ ഒരു റാപ്പർ **അല്ല**. പങ്കുവെച്ച കാനോണിക്കൽ `action_digest`, അംഗീകാരത്തിന്റെ നയ/കീ/കാലാവധി **പുതുക്കലുകൾ**, അംഗീകാരം **ഒറ്റസമയം ഉപയോഗം** എന്നിവ ഇവിടെ പരിശോധിക്കപ്പെടുന്നു, ഇപ്പോൾ *നടപ്പിലാക്കുന്ന* പ്രവൃത്തിയുമായി സമന്വയിപ്പിച്ച്.

ഓരോ പരാജയവും വ്യത്യസ്ത കാരണത്താൽ നിരസിക്കുന്നു, അതിനാൽ നിരസിക്കൽ വായിക്കുന്നവൻ അധികാരം പഴകിയിട്ടുണ്ടോ (നയം മാറി, കീ മാറ്റി, അംഗീകാരം കാലഹരണപ്പെട്ടു, അംഗീകാരം ഉപയോഗിച്ചേക്കും) അല്ലെങ്കിൽ അവലംബമുണ്ട് എന്ന നിലയിൽ സാധുവായ അംഗീകാരത്തേക്കാൾ പ്രവർത്തി മാറ്റപ്പെട്ടിട്ടുണ്ടോയെന്ന് (ഡൈജസ്റ്റ് മാറ്റം) പറയാം.


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## ബൈൻഡിങ്ങ് പിടിച്ചെടുക്കുന്നത് എന്ത് ആണ്

താഴെ കാണിക്കുന്ന ഓരോ കേസ് **ഡിഫർന്റ് കാരണം** കൊണ്ട് **ക്ലോസ്ഡ്** ആയി ഫെയ്ൽ ചെയ്യും. ആദ്യ ബ്ലോക്ക് ക്ലാസിക് സെറ്റ് ആണ് (ടാമ്പർ, കഫ്യൂസ്ഡ് ഡെപ്യൂട്ടി, റീപ്ലേ, ഏതൊരു অথോറിറ്റിയിലോ ഫോർജറി, തെറ്റായ ഇൻപുട്ട്). രണ്ടാമത്തെ ബ്ലോക്ക് അവകാശം സത്യവാങ്മൂലിയാക്കുന്ന ആദ്യത്തെ ജോഡി ആണ്:

- **സ്റ്റെയിൽ অথോറിറ്റി** — സിഗ്നേച്ചർ ഇപ്പോഴും válido ആണെങ്കിലും, പോളിസി വേർഷൻ മാറ്റം സംഭവിച്ചു, അപ്‌പ്രൂവർ കീ പിന് ചെയ്ത രജിസ്ട്രിയിൽ നിന്നു മാറ്റപ്പെട്ടു, അല്ലെങ്കിൽ അപ്‌പ്രൂവ് ചെയ്യൽ നിർവഹണത്തിന് മുമ്പ് കാലഹരണപ്പെട്ടു;
- **ഡൈജസ്റ്റ് സബ്സ്റ്റിറ്റ്യൂഷൻ** — ഒരു വാലിഡ് ആയി ഒപ്പ് വെച്ച ആക്ഷൻ റസീപ്പ്, അതിന്റെ `parent_approval_ref` ആകെയുള്ളത് ഒരു *യഥാർത്ഥ* അപ്‌പ്രൂവലിനെ സൂചിപ്പിക്കുന്നതാണ്, എങ്കിലും ആ അപ്‌പ്രൂവലിന്റെ കാനോണിക്കൽ ആക്ഷൻ ഡൈജസ്റ്റ് യഥാർത്ഥ നിർവഹിക്കുന്ന ആക്ഷനുമായി യോജിച്ചില്ല.


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## ഇത് തെളിയിക്കുന്നതും — ഇതു തെളിയിക്കാത്തതും

**തെളിയിക്കുന്നു:** ഒരു പേര് വ്യക്തമാക്കിയ മനുഷ്യൻ *ഈ കൃത്യCanonical പ്രവർത്തനം* (പൂർണ്ണ പ്രവർത്തനവും ഡൈജസ്റ്റ്‌വും, പിന്ഡ് രജിസ്ട്രിയിൽ നിന്നുള്ള കീ ഉപയോഗിച്ച് ഒപ്പ് വെച്ചത്) അംഗീകരിച്ചിട്ടുണ്ടെന്ന്, ഏജൻറ് *അടുത്തുള്ള അംഗീകൃത പ്രവർത്തനം* കൃത്യമായി (അതിന് സമാനമായ ഡൈജസ്റ്റ്‌, അംഗീകാരത്തിനൊപ്പം `receipt_hash` വഴി ബന്ധപ്പെട്ട റെസീറ്റ്, പാഠത്തിന്റെ സ്വന്തം ചെൻ റീതി) നടപ്പാക്കിയതെന്ന് — അംഗീകാരം വന്ന നയം പതിപ്പും കീയും കാലഹരണപ്പെട്ടിട്ടില്ലാതെയും നിലവിലുണ്ടായിരുന്ന പോലെ, കൃത്യമായി ഒരിക്കൽ. ഏതെങ്കിലും പാർശ്വം മാറിയാൽ, ചെയിൻേ് ബന്ദിച്ചുപോകും, അപ്രത്യക്ഷരചന കാരണം **ഏത്** സ്വഭാവം തകർന്നുവെന്നറിയിക്കും: കാലഹരണപ്പെട്ട അധികാരവുമായോ പ്രവർത്തനം മാറിയതുമായോ.

**തെളിയിക്കാത്തത്:** അംഗീകാരം_ui മനുഷ്യന് അവർ ഒപ്പുവെക്കാമെന്ന് കരുതുന്നതും കാണിച്ചിട്ടുണ്ടെന്ന് (WYSIWYS അതിന്റെ സ്വന്തം പ്രശ്നമാണ്), കീ റൊട്ടേഷൻ മുൻപ് തട്ടിക്കൊണ്ടുപോയോ നിർബന്ധപ്പെടുത്തി പോയോ അല്ലെന്ന്, അല്ലെങ്കിൽ താഴ്ന്ന ഫലങ്ങൾ പ്രവർത്തനത്തോട് പൊരുത്തപ്പെട്ടു എന്നും. ഒപ്പിട്ടതു = അനുമതിയുള്ളത് അല്ല: പഴകിയ നയം, റൊട്ടേറ്റായ കീ, കാലഹരണപ്പെട്ട വിൻഡോ, അല്ലെങ്കിൽ വ്യത്യസ്ത ഡൈജസ്റ്റ് എന്നിവയിൽ സാധുവായ ഒപ്പ് ഇതുവരെയൊന്നും നൽകുന്നില്ല.

രണ്ട് റെസീറ്റ് തരം പാഠത്തിന്റെ ലോപ്‌ഫോൾഡിനെയും ഒരേ `verify_chain` കോഡ് പാതയെയും പങ്കിടുന്നു ഉദ്ദേശ്യപൂർവ്വം: പ്രധാന നോട്ട്‌ബുക്കിലെ പ്രവർത്തന റെസീറ്റുകൾക്കായുള്ള ബൈൻഡിംഗ് ആണ് മനുഷ്യൻ നൽകിയ അംഗീകാരം പരിശോധിക്കുന്നതുമായാകെയുള്ള കോഡ്. ഒരു പരിശോധനാ കരാർ, വേർതിരിച്ചിട്ടുള്ള പിന്‍ അധികാരങ്ങൾ, കനീനിക്കൽ പ്രവർത്തന ഡൈജസ്റ്റ് എന്നിവ കൊണ്ട് ചേർന്നതാണ്.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
